# 04 主效果估计

- **口径**：转化分析只用锁定的 **23 天 outcome 窗（2014-10-11→11-02）**，分母为窗内 Clicks；质量检查用 37 天窗。
- **主分析**：双样本比例 **Z 检验**（检验统计量用 pooled 方差，95% CI 用 unpooled 方差）。
- **稳健性复核（仅辅助证据）**：以**天为单位**的 cluster bootstrap（B=10,000，seed 来自 config）与 **Delta Method** 解析 SE；日数仅 23 天，bootstrap CI 只作辅助。
- Payments/Enrollments 只作业务辅助诊断，**不是独立主假设检验**。
- 原则：**"未达统计显著"不等于"没有差异/无影响"**，必须结合效应量、CI、精度、MDE 综合解释。

In [1]:
# 加载：锁定参数 + 23 天 outcome 窗
import json, tomllib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
CFG = tomllib.load(open(ROOT/"config"/"analysis_config.toml","rb"))
LK, EXP = CFG["locked"], CFG["exploratory"]
long = pd.read_csv(ROOT/"data/processed/daily_long.csv", parse_dates=["Date"])
ow = long[long["OutcomeComplete"]].copy()
assert ow["Date"].nunique() == LK["windows"]["outcome_days"] == 23
B = EXP["bootstrap_b"]; SEED = EXP["random_seed"]
print("outcome days:", ow["Date"].nunique(), "| bootstrap B =", B, "| seed =", SEED)

outcome days: 23 | bootstrap B = 10000 | seed = 20260831


## outcome 窗总量汇总

In [2]:
# 两组 Clicks/Enrollments/Payments 总量
tot = ow.groupby("Group")[["Clicks","Enrollments","Payments"]].sum().astype(int)
tot["GrossConversion"] = tot["Enrollments"]/tot["Clicks"]
tot["NetConversion"] = tot["Payments"]/tot["Clicks"]
tot["PayPerEnrollment"] = tot["Payments"]/tot["Enrollments"]
print(tot.to_string(float_format=lambda x: f"{x:.6f}"))

            Clicks  Enrollments  Payments  GrossConversion  NetConversion  PayPerEnrollment
Group                                                                                      
Control      17293         3785      2033         0.218875       0.117562          0.537120
Experiment   17260         3423      1945         0.198320       0.112688          0.568215


## Gross / Net 双样本比例 Z 检验（主分析）
对每个指标报告：对照率、实验率、绝对差(E−C)、相对变化、Z、双侧 p、95% CI（unpooled）。

In [3]:
# 双样本比例检验：检验用 pooled 方差，CI 用 unpooled 方差
def two_prop_test(x_c, n_c, x_e, n_e, alpha=LK["alpha_two_sided"]):
    pc, pe = x_c/n_c, x_e/n_e
    d = pe - pc
    p_pool = (x_c + x_e)/(n_c + n_e)
    se_pool = np.sqrt(p_pool*(1-p_pool)*(1/n_c + 1/n_e))
    z = d/se_pool
    pval = 2*stats.norm.sf(abs(z))
    se_un = np.sqrt(pc*(1-pc)/n_c + pe*(1-pe)/n_e)
    za = stats.norm.ppf(1-alpha/2)
    return {"rate_control": pc, "rate_experiment": pe, "abs_diff": d,
            "relative_change": d/pc, "z": float(z), "p_value": float(pval),
            "se_test_pooled": float(se_pool), "se_ci_unpooled": float(se_un),
            "ci95_low": float(d-za*se_un), "ci95_high": float(d+za*se_un)}

z_results = {}
specs = [("GrossConversion", "Enrollments"), ("NetConversion", "Payments")]
for name, num in specs:
    g = {grp: tot.loc[grp] for grp in ["Control","Experiment"]}
    r = two_prop_test(int(g["Control"][num]), int(g["Control"]["Clicks"]),
                      int(g["Experiment"][num]), int(g["Experiment"]["Clicks"]))
    z_results[name] = r
    sig = "显著(拒绝H0)" if r["p_value"] < LK["alpha_two_sided"] else "不显著(不能拒绝H0)"
    print(f"=== {name} ===")
    print(f" Control={r['rate_control']:.6f}  Experiment={r['rate_experiment']:.6f}")
    print(f" abs diff(E-C)={r['abs_diff']:+.6f} ({r['abs_diff']*100:+.3f}pp)  relative={r['relative_change']:+.2%}")
    print(f" Z={r['z']:.4f}  p={r['p_value']:.6g}  -> {sig}")
    print(f" 95% CI(unpooled)=({r['ci95_low']:+.6f},{r['ci95_high']:+.6f})")

=== GrossConversion ===
 Control=0.218875  Experiment=0.198320
 abs diff(E-C)=-0.020555 (-2.055pp)  relative=-9.39%
 Z=-4.7018  p=2.5784e-06  -> 显著(拒绝H0)
 95% CI(unpooled)=(-0.029120,-0.011990)
=== NetConversion ===
 Control=0.117562  Experiment=0.112688
 abs diff(E-C)=-0.004874 (-0.487pp)  relative=-4.15%
 Z=-1.4192  p=0.155841  -> 不显著(不能拒绝H0)
 95% CI(unpooled)=(-0.011604,+0.001857)


## Payments/Enrollments 辅助业务诊断（非独立主检验）
Gross=Enr/Click、Net=Pay/Click 已在数学上决定 Pay/Enr = Net/Gross，为避免冗余检验，这里只报点估计、相对变化与方向，CI 仅作描述性参考、不作为主假设检验。

In [4]:
# 试听用户质量辅助指标
ppe_c = tot.loc["Control","Payments"]/tot.loc["Control","Enrollments"]
ppe_e = tot.loc["Experiment","Payments"]/tot.loc["Experiment","Enrollments"]
se_ppe = np.sqrt(ppe_c*(1-ppe_c)/tot.loc["Control","Enrollments"] +
                 ppe_e*(1-ppe_e)/tot.loc["Experiment","Enrollments"])
d_ppe = ppe_e-ppe_c
ppe_aux = {"control": float(ppe_c), "experiment": float(ppe_e), "abs_diff": float(d_ppe),
           "relative_change": float(d_ppe/ppe_c), "direction": "up" if d_ppe > 0 else "down",
           "descriptive_ci95": [float(d_ppe-1.96*se_ppe), float(d_ppe+1.96*se_ppe)],
           "note": "auxiliary business diagnostic only; NOT an independent primary hypothesis test"}
print(f"Pay/Enroll: Control={ppe_c:.6f}, Experiment={ppe_e:.6f}")
print(f" diff={d_ppe:+.6f} ({d_ppe*100:+.3f}pp), relative={d_ppe/ppe_c:+.2%}, direction={ppe_aux['direction']}")
print("描述性95%CI(非主检验):", tuple(round(x,6) for x in ppe_aux["descriptive_ci95"]))

Pay/Enroll: Control=0.537120, Experiment=0.568215
 diff=+0.031095 (+3.109pp), relative=+5.79%, direction=up
描述性95%CI(非主检验): (0.008123, 0.054066)


## Day-cluster Bootstrap（B=10,000，辅助证据）
以**天**为重采样单位：每组有放回抽 23 天 → 用抽中天数的 Σ分子/Σ分母重算比率 → 差值；得到 percentile CI。
**局限声明：日级聚合数据的敏感性分析，不等价 user-level bootstrap；仅 23 个 cluster，CI 仅作辅助。**

In [5]:
# day-cluster bootstrap
rng = np.random.default_rng(SEED)
piv = ow.pivot(index="Date", columns="Group")
n_days = ow["Date"].nunique()

def daily_arrays(num):
    return (piv[num]["Control"].to_numpy(float), piv["Clicks"]["Control"].to_numpy(float),
            piv[num]["Experiment"].to_numpy(float), piv["Clicks"]["Experiment"].to_numpy(float))

def boot_diff(num, b=B):
    yc, xc, ye, xe = daily_arrays(num)
    diffs = np.empty(b)
    idx_c = rng.integers(0, n_days, size=(b, n_days))
    idx_e = rng.integers(0, n_days, size=(b, n_days))
    for i in range(b):
        rc = yc[idx_c[i]].sum()/xc[idx_c[i]].sum()
        re_ = ye[idx_e[i]].sum()/xe[idx_e[i]].sum()
        diffs[i] = re_-rc
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return diffs, float(lo), float(hi), float(diffs.mean())

boot = {}
for name, num in [("GrossConversion","Enrollments"), ("NetConversion","Payments"),
                  ("PayPerEnrollment","Payments")]:
    if name == "PayPerEnrollment":
        # Pay/Enr 的分母是 Enrollments，单独构造
        yc = piv["Payments"]["Control"].to_numpy(float); dc = piv["Enrollments"]["Control"].to_numpy(float)
        ye = piv["Payments"]["Experiment"].to_numpy(float); de = piv["Enrollments"]["Experiment"].to_numpy(float)
        diffs = np.empty(B)
        ic = rng.integers(0,n_days,size=(B,n_days)); ie = rng.integers(0,n_days,size=(B,n_days))
        for i in range(B):
            diffs[i] = ye[ie[i]].sum()/de[ie[i]].sum() - yc[ic[i]].sum()/dc[ic[i]].sum()
    else:
        diffs, _, _, _ = boot_diff(num)
    lo, hi = np.percentile(diffs,[2.5,97.5])
    boot[name] = {"ci95_low": float(lo), "ci95_high": float(hi), "boot_mean": float(diffs.mean())}
    print(f"{name:18s} bootstrap percentile CI=({lo:+.6f},{hi:+.6f}), mean={diffs.mean():+.6f}")

GrossConversion    bootstrap percentile CI=(-0.045871,+0.004770), mean=-0.020625
NetConversion      bootstrap percentile CI=(-0.022563,+0.011819), mean=-0.004868


PayPerEnrollment   bootstrap percentile CI=(-0.034030,+0.095545), mean=+0.031203


## Delta Method（ratio 指标解析 SE，以天为独立 cluster）
R=ΣY/ΣX。按 delta method：Var(R) ≈ s²(Y−R·X)/(n·X̄²)，两组独立故差值 SE=√(Var_C+Var_E)。

In [6]:
# cluster delta method
def delta_ratio_var(df_g, num, den):
    n = len(df_g); R = df_g[num].sum()/df_g[den].sum()
    resid = df_g[num] - R*df_g[den]
    s2 = (resid**2).sum()/(n-1)
    return R, s2/(n*df_g[den].mean()**2)

delta = {}
for name, num, den in [("GrossConversion","Enrollments","Clicks"),
                       ("NetConversion","Payments","Clicks"),
                       ("PayPerEnrollment","Payments","Enrollments")]:
    Rc, Vc = delta_ratio_var(ow[ow.Group=="Control"], num, den)
    Re, Ve = delta_ratio_var(ow[ow.Group=="Experiment"], num, den)
    se = np.sqrt(Vc+Ve); d = Re-Rc
    delta[name] = {"point": float(d), "se": float(se),
                   "ci95_low": float(d-1.96*se), "ci95_high": float(d+1.96*se)}
    print(f"{name:18s} point={d:+.6f} SE_delta={se:.6f} CI=({d-1.96*se:+.6f},{d+1.96*se:+.6f})")

GrossConversion    point=-0.020555 SE_delta=0.013243 CI=(-0.046512,+0.005402)
NetConversion      point=-0.004874 SE_delta=0.008861 CI=(-0.022241,+0.012494)
PayPerEnrollment   point=+0.031095 SE_delta=0.033723 CI=(-0.035003,+0.097192)


## 三法对照（Z 为主，Delta/Bootstrap 为稳健性复核）

In [7]:
# 对照表
rows = []
for name in ["GrossConversion","NetConversion","PayPerEnrollment"]:
    if name in z_results:
        z = z_results[name]; point, se = z["abs_diff"], z["se_ci_unpooled"]
        ci_z = (z["ci95_low"], z["ci95_high"]); zstat, pp = z["z"], z["p_value"]
    else:
        point, se, ci_z, zstat, pp = ppe_aux["abs_diff"], None, tuple(ppe_aux["descriptive_ci95"]), None, None
    dl, bt = delta[name], boot[name]
    rows.append({"Metric": name, "point(E-C)": point,
                 "Z_SE(unpooled)": se, "Z_95CI": ci_z, "Z": zstat, "p": pp,
                 "Delta_SE": dl["se"], "Delta_95CI": (dl["ci95_low"], dl["ci95_high"]),
                 "Boot_95CI": (bt["ci95_low"], bt["ci95_high"])})
cmp_df = pd.DataFrame(rows)
for _, r in cmp_df.iterrows():
    print(f"{r['Metric']}: point={r['point(E-C)']:+.6f}")
    print(f"  Z(主)        CI=({r['Z_95CI'][0]:+.6f},{r['Z_95CI'][1]:+.6f})  Z={r['Z']} p={r['p']}")
    print(f"  Delta(复核)  CI=({r['Delta_95CI'][0]:+.6f},{r['Delta_95CI'][1]:+.6f})")
    print(f"  Boot(辅助)   CI=({r['Boot_95CI'][0]:+.6f},{r['Boot_95CI'][1]:+.6f})")
print('''
差异来源：Z 检验把每次 click 视为独立 Bernoulli；Delta/Bootstrap 以“天”为 cluster，
承认日转化率的日间波动（过度离散），cluster SE 约为 iid SE 的 2.6-3.0 倍，且只有 23 个 cluster，
因此区间显著更宽——Gross 的 cluster 区间触及/跨过 0。点估计与方向三法一致，但显著性强弱不同：
按事前指定，Z 检验为主分析；cluster 法作为辅助，额外揭示“只有23天”带来的不确定性（写入 Limitations）。''')

GrossConversion: point=-0.020555
  Z(主)        CI=(-0.029120,-0.011990)  Z=-4.701830023753982 p=2.578401033720593e-06
  Delta(复核)  CI=(-0.046512,+0.005402)
  Boot(辅助)   CI=(-0.045871,+0.004770)
NetConversion: point=-0.004874
  Z(主)        CI=(-0.011604,+0.001857)  Z=-1.4192001144365733 p=0.15584068262150205
  Delta(复核)  CI=(-0.022241,+0.012494)
  Boot(辅助)   CI=(-0.022563,+0.011819)
PayPerEnrollment: point=+0.031095
  Z(主)        CI=(+0.008123,+0.054066)  Z=nan p=nan
  Delta(复核)  CI=(-0.035003,+0.097192)
  Boot(辅助)   CI=(-0.034030,+0.095545)

差异来源：Z 检验把每次 click 视为独立 Bernoulli；Delta/Bootstrap 以“天”为 cluster，
承认日转化率的日间波动（过度离散），cluster SE 约为 iid SE 的 2.6-3.0 倍，且只有 23 个 cluster，
因此区间显著更宽——Gross 的 cluster 区间触及/跨过 0。点估计与方向三法一致，但显著性强弱不同：
按事前指定，Z 检验为主分析；cluster 法作为辅助，额外揭示“只有23天”带来的不确定性（写入 Limitations）。


## 正确解释"不显著"
- Gross：效应 −2.06pp，Z 主检验 CI 全负、p<0.001、降幅大于锁定 MDE=1pp。**同一口径：三法点估计与方向一致，但显著性强度依赖 click 相互独立假设——day-cluster（Delta/Bootstrap）口径下 Gross 的 CI 跨 0**。因此规范表述为"在事前指定的 Z 主分析下筛选显著减少注册；在以天为 cluster 的口径下证据更弱"，不得写成无条件显著。
- Net：效应 −0.49pp、CI 跨 0、p>0.05 → 只是**"现有精度下未能拒绝 H0"**，不是"证明无影响"；其 CI 同时包含 0 和有业务意义的负向效应，且前文已表明实验对 0.75pp 级别效应 underpowered，必须结合 CI 宽度与 MDE 解读，最终是否可接受由相应阶段非劣效判定。
- 补充：以天为 cluster 的 Delta/bootstrap 区间更宽（23 个 cluster + 日间过度离散），Gross 在 cluster 口径下区间触 0；这提示主结论的显著性强度依赖"click 相互独立"的前提，须在 Limitations 说明，但不改变事前指定的 Z 主分析结论。

In [8]:
# 主效果森林图（零点线；Net 预留 -δ 非劣效边界线，正式判定留待非劣效环节）
fig, ax = plt.subplots(figsize=(9, 4.2), dpi=150)
order = ["GrossConversion","NetConversion","PayPerEnrollment"]
labels = {"GrossConversion":"Gross Conversion (Enr/Click)",
          "NetConversion":"Net Conversion (Pay/Click) [core]",
          "PayPerEnrollment":"Pay/Enrollment (auxiliary)"}
for i, name in enumerate(order):
    z = z_results.get(name); a = ppe_aux if name=="PayPerEnrollment" else z
    pt = a["abs_diff"]; lo, hi = a["ci95_low"] if name!="PayPerEnrollment" else a["descriptive_ci95"][0], a["ci95_high"] if name!="PayPerEnrollment" else a["descriptive_ci95"][1]
    color = "#1f77b4" if name!="PayPerEnrollment" else "#7f7f7f"
    ax.errorbar(pt*100, i, xerr=[[(pt-lo)*100],[(hi-pt)*100]], fmt="o", color=color, capsize=5, lw=2)
    ax.text(hi*100+0.08, i, f"{pt*100:+.2f}pp [{lo*100:+.2f}, {hi*100:+.2f}]", va="center", fontsize=9)
ax.axvline(0, color="black", lw=1.2)
ax.axvline(-LK["ni_delta_net"]*100, color="#d62728", ls="--", lw=1.2)
ax.text(-LK["ni_delta_net"]*100+0.05, 0.35, "-delta=-0.75pp (NI boundary)",
        color="#d62728", fontsize=8, ha="left")
ax.set_yticks(range(len(order))); ax.set_yticklabels([labels[n] for n in order])
ax.invert_yaxis(); ax.set_xlabel("Absolute difference (Experiment - Control), percentage points")
ax.set_title("Main effect estimates with 95% CI (Z-test primary; Pay/Enr descriptive)")
ax.grid(axis="x", alpha=.3)
fig.tight_layout(); fig.savefig(ROOT/"reports"/"figures"/"fig_forest_effects.png"); plt.close(fig)
print("saved reports/figures/fig_forest_effects.png")

saved reports/figures/fig_forest_effects.png


In [9]:
# 落盘主效果结果 + 回读验证
out = {"window": "23-day outcome window", "totals": tot.reset_index().to_dict(orient="records"),
       "z_tests": z_results, "ppe_aux": ppe_aux,
       "bootstrap_B": B, "bootstrap": boot, "delta": delta}
p = ROOT/"data"/"processed"/"main_effects.json"
p.write_text(json.dumps(out, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
back = json.loads(p.read_text(encoding="utf-8"))
assert abs(back["z_tests"]["GrossConversion"]["z"] - z_results["GrossConversion"]["z"]) < 1e-12
print("main_effects.json written & re-read OK")

main_effects.json written & re-read OK


## 小结
1. **Gross Conversion 在 Z 主分析下显著下降**：−2.06pp（−9.39%），Z 检验 95%CI 全负且超出锁定 MDE=1pp；**同一口径：三法点估计与方向一致，但显著性强度依赖 click 独立假设，day-cluster 口径下 Gross CI 跨 0，不表述为无条件显著**（与 README Limitations 一致）。
2. **Net Conversion 点估 −0.49pp（−4.15%），不显著**：CI 跨 0；这是"精度不足以下结论"而非"无差异"，Delta/Bootstrap 区间方向一致，最终由非劣效框架裁决。
3. **Pay/Enroll +3.11pp（+5.79%）**：进入付费的注册用户占比上升，与"筛掉低意愿注册、留下更高质量用户"的业务假设方向一致，仅作辅助诊断。
4. **方法间一致性与张力**：三法点估计与方向一致；但以天为 cluster 的 Delta/bootstrap 承认日间过度离散（SE 为 iid 的 2.6–3.0 倍、仅 23 个 cluster），区间明显更宽，Gross 的 cluster 区间触及/跨过 0。按事前锁定，**Z 检验是主分析**（Gross 显著下降、Net 不显著）；cluster 法不推翻方向，但额外标注了有限天数下的不确定性，将在 README Limitations 中讨论，不得只报对结论有利的窄区间。